# Earnings-Season-Tide — a quantitative teardown 🔬
### Calendar contrast · HAC inference · block-bootstrap CI · selection · excess-vs-excess capacity

![Signal: None](https://img.shields.io/badge/Signal-None-c0392b?style=flat-square)
![Tradability: Mirage](https://img.shields.io/badge/Tradability-Mirage-c0392b?style=flat-square)
![Separable tide%3F: Not supported](https://img.shields.io/badge/Separable_tide%3F-Not_supported-8b949e?style=flat-square)

The deep companion to the [notebook for the curious](01_for_the_curious.ipynb) — *same seven beats, every claim now carrying its standard error.* We measure the aggregate-index drift inside four pre-declared peak-earnings windows, put a Newey-West *t* on the in-window minus out-of-window difference, bound it with a circular block bootstrap, check the per-window selection trap, and race a calendar overlay excess-of-cash against buy-and-hold.

> ⚠️ **Not investment advice.** Data: SPY total-return daily bars (cache-first; synthetic fallback offline). Methods: HAC/Newey-West, block bootstrap, excess-vs-excess Sharpe. Literature in [`docs/references.md`](../docs/references.md).
>
> 💡 **The `💡 In plain words` notes** translate each result back into intuition. House style in [METHODOLOGY.md](../../../METHODOLOGY.md).

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))          # the study package
sys.path.insert(0, os.path.abspath("../../.."))    # repo root (quantlab/)
%matplotlib inline
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.figsize": (9.5, 5.0), "axes.grid": True,
                     "grid.alpha": .3, "axes.spines.top": False, "axes.spines.right": False})
RED, AMBER, GREEN, GREY = "#c0392b", "#dab617", "#2ea44f", "#8b949e"
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

from earnings_season_tide import data, strategy as st

# Frozen real-tape headline numbers (mirror of ../docs/results.md, as-of 2026-06-18).
R = {'as_of': '2026-06-18', 'fingerprint': 'ff2579c16a7d', 'span': '1993-01-29 → 2025-12-31', 'n_days': 8288, 'n_in': 1737, 'n_out': 6550, 'mean_in_bps': 6.86, 'mean_out_bps': 3.28, 'diff_bps': 3.59, 'ann_in_pct': 17.3, 'ann_out_pct': 8.3, 'tstat_diff': 1.27, 'ci_lo': -1.67, 'ci_hi': 9.17, 'jan_bps': 0.37, 'jan_t': -0.72, 'apr_bps': 9.15, 'apr_t': 1.23, 'jul_bps': 5.44, 'jul_t': 0.33, 'oct_bps': 11.16, 'oct_t': 1.39, 'overlay_ann': 3.54, 'overlay_sr': 0.4, 'overlay_tim': 0.21, 'overlay_turnover': 8.0, 'buyhold_ann': 10.15, 'buyhold_sr': 0.54, 'ctrl_null_diff': -0.71, 'ctrl_null_t': -0.27, 'ctrl_tide_diff': 11.29, 'ctrl_tide_t': 4.24, 'ctrl_tide_bps': 12}

# Cache-first real tape, synthetic fallback. We BANNER which tape every cell below runs on.
TAPE = "SYNTHETIC"
try:
    bars = data.load_real("SPY", "total_return", fetch=False)
    bars = bars[bars.index <= "2025-12-31"]          # drop the in-progress year
    TAPE = "REAL"
    print(f"REAL tape: SPY total-return, {bars.index.min().date()} -> {bars.index.max().date()}, "
          f"{len(bars)} days, fingerprint {data.fingerprint(bars)}")
except Exception as e:
    bars, _ = data.synthetic_daily(n_years=33, tide_bps=0.0, seed=321)
    print(f"OFFLINE -> SYNTHETIC null tape ({len(bars)} days). Reason: {e}")
    print("Headline real-tape numbers are quoted from the frozen R dict (docs/results.md).")
print("TAPE =", TAPE)


REAL tape: SPY total-return, 1993-01-29 -> 2025-12-31, 8288 days, fingerprint ff2579c16a7d
TAPE = REAL


## Verdict, up front

| Axis | Stamp | Why |
|---|---|---|
| Signal | **None** | in−out diff **+3.59 bps/day**, HAC *t* **+1.27**; bootstrap CI **[−1.67, +9.17]** includes 0; no window clears *t*=2 |
| Tradability | **Mirage** | overlay **+3.5%/yr, SR 0.40** vs buy-hold **+10.2%/yr, SR 0.54**; 79% in cash |
| Separable tide? | **Not supported** | PEAD is single-stock & surprise-conditioned; it does not aggregate into a directional index seasonal |

> 💡 **In plain words:** the index drifts marginally faster during earnings weeks, but it's the ordinary up-drift showing through a calendar slice — not a separable, tradable tide.

## 1 · The claim, steelmanned

- **H₁ (the tide).** Mean daily index return is higher in-window than out: `E[r | in] − E[r | out] > 0`, robustly (HAC *t* ≥ 2).
- **H₂ (per-window).** At least one of the four windows individually carries a drift clearing *t* = 2 (and survives the four-way selection).
- **H₃ (tradable).** A long-in-window/cash overlay beats buy-and-hold on *excess-of-cash* Sharpe after one-way costs.

We reject H₁–H₃ if the differences sit inside their error bars and the overlay trails the index.

## 2 · So what? — what rides on each answer

A +3.6 bps/day in-window premium over ~1,740 in-window days is ~+6.2% of cumulative log-drift parked in 21% of the calendar — *if* real, an attractive risk-budget deployment. The stakes are entirely in the standard error: a gap of this size is exactly what ordinary equity drift produces over any 21% slice of days, so the question is whether the *difference* is separable from baseline drift, not whether in-window days are positive (they are — so are most days).

## 3 · How we'd know — the protocol

1. **Measure** the in/out contrast on log returns (no fitting).
2. **Robust inference** — regress `r` on the in-window dummy with a Newey-West HAC covariance; the slope is the mean difference, its HAC SE gives the *t*. Bound the difference with a **circular block bootstrap** (block = 21 ≈ one month, preserving vol clustering and calendar alignment).
3. **Critique magnitude** — split by window to expose the selection trap (best of four pre-chosen windows).
4. **Alpha vs beta** — the in-window return is *mostly market beta you already hold*; the overlay's contribution is the only thing that isn't.
5. **Capacity** — excess-vs-excess Sharpe race, costs one-way × NAV, time-in-market.

> 💡 **In plain words:** a t-test that respects autocorrelation, a resampling error bar that respects volatility clusters, and a fair race that doesn't reward sitting in cash.

## 4 · The teardown

### 4.1 · The contrast and its HAC *t*

In [2]:
c = st.calendar_contrast(bars)
lo, hi = st.block_bootstrap_ci(bars, block=21, n_boot=2000, seed=321)
tab = pd.DataFrame({
    'group': ['in-window','out-of-window','difference (in-out)'],
    'mean_bps': [c['mean_in_bps'], c['mean_out_bps'], c['diff_bps']],
    'n': [c['n_in'], c['n_out'], np.nan],
})
display(tab)
print(f'HAC t on difference : {c["tstat_diff"]:+.3f}')
print(f'block-bootstrap 95% CI on difference : [{lo:+.2f}, {hi:+.2f}] bps/day')
print(f'TAPE = {TAPE}')
if TAPE != 'REAL':
    print(f'[banner] SYNTHETIC null. REAL: diff {R["diff_bps"]:+.2f}, t {R["tstat_diff"]:+.2f}, '
          f'CI [{R["ci_lo"]:+.2f}, {R["ci_hi"]:+.2f}]')


,group,mean_bps,n
0,in-window,6.861,"1,737.000"
1,out-of-window,3.275,"6,550.000"
2,difference (in-out),3.586,NaN


HAC t on difference : +1.272
block-bootstrap 95% CI on difference : [-1.67, +9.17] bps/day
TAPE = REAL


> 💡 **In plain words:** the gap is positive but its *t* is well under 2 and the interval crosses zero — by the desk's inference bar, that is **not** a real signal.

### 4.2 · The per-window selection trap

In [3]:
pw = st.per_window_means(bars)
display(pw)
best = pw.loc[pw['mean_bps'].idxmax()]
print(f'Strongest window: {best["window"]} at {best["mean_bps"]:+.2f} bps/day, '
      f't={best["tstat"]:+.2f}')
print('None of the four clears |t|>=2 -> picking the best of four would be pure snooping.')


,window,n_days,mean_bps,tstat
0,Jan (Q4),361,0.367,-0.716
1,Apr (Q1),436,9.150,1.225
2,Jul (Q2),471,5.437,0.325
3,Oct (Q3),469,11.162,1.392


Strongest window: Oct (Q3) at +11.16 bps/day, t=+1.39
None of the four clears |t|>=2 -> picking the best of four would be pure snooping.


### 4.3 · Alpha vs beta — what's actually new

In [4]:
# The in-window days are mostly the market you already own. The overlay's only
# distinct contribution is the in/out timing; everything else is beta.
r = st.log_returns(bars)
ann_all = r.mean()*252*100
print(f'Whole-sample drift: {ann_all:+.2f}%/yr (this is the beta you hold either way)')
print(f'In-window annualised: {c["ann_in_pct"]:+.2f}%/yr  -- higher, but it IS beta,'
      ' just sampled on a calendar.')


Whole-sample drift: +10.15%/yr (this is the beta you hold either way)
In-window annualised: +17.29%/yr  -- higher, but it IS beta, just sampled on a calendar.


### 4.4 · Capacity — the excess-vs-excess race

In [5]:
rows = []
for cost in (0.0, 1.0, 2.0, 5.0):
    rc = st.race_vs_buyhold(bars, cost_bps=cost)
    rows.append({'cost_bps': cost, 'overlay_ann%': rc['overlay_ann_pct'],
                 'overlay_SR': rc['overlay_sharpe_net'],
                 'buyhold_ann%': rc['buyhold_ann_pct'],
                 'buyhold_SR': rc['buyhold_sharpe'],
                 'time_in_mkt': rc['time_in_market'],
                 'turnover/yr': rc['turnover_per_yr']})
sweep = pd.DataFrame(rows)
display(sweep)
print('Turnover is one-way x NAV. The overlay trails buy-and-hold at every cost level.')


,cost_bps,overlay_ann%,overlay_SR,buyhold_ann%,buyhold_SR,time_in_mkt,turnover/yr
0,0.000,3.624,0.409,10.147,0.544,0.210,7.967
1,1.000,3.544,0.400,10.147,0.544,0.210,7.967
2,2.000,3.465,0.391,10.147,0.544,0.210,7.967
3,5.000,3.226,0.365,10.147,0.544,0.210,7.967


Turnover is one-way x NAV. The overlay trails buy-and-hold at every cost level.


> 💡 **In plain words:** even at zero cost the overlay loses the race — it isn't a cost problem, it's that you're voluntarily sitting out 79% of the up-drift to chase a gap that isn't there.

## 5 · The verdict

- **Signal — None.** in−out **+3.59 bps/day**, HAC *t* **+1.27**, bootstrap CI **[-1.67, +9.17]** includes 0; H₁ and H₂ rejected.
- **Tradability — Mirage.** overlay **+3.54%/yr, SR 0.40** vs buy-hold **+10.15%/yr, SR 0.54**; H₃ rejected.
- **Separable tide? — Not supported.** PEAD aggregates away at the index level.

## 6 · Could you trade it?

Break-even is moot: the edge is statistically zero and the overlay loses the race at **zero** cost, so there is no cost budget to spend. The binding facts are **time-in-market 21%** and an excess-of-cash Sharpe **below** buy-and-hold. No square-root-impact capacity analysis is warranted for a non-edge.

## 7 · Going further

- **Surprise-conditioned aggregation.** Re-run conditioning on the sign of aggregate earnings surprise (needs forward earnings data; drops the pure-calendar property).
- **Second moment.** Test an in-window *variance* tide — earnings season may move vol even where it doesn't move drift.
- **Snooping correction.** Sweep all plausible window definitions under a Sullivan-Timmermann-White Reality Check instead of fixing four — the honest way to handle the believer's 'but my windows are different' objection.

*Synthetic positive control (the harness is a faithful detector):*

In [6]:
# The same machinery recovers a planted tide and reads the null as null.
b0,_ = data.synthetic_daily(n_years=33, tide_bps=0.0, seed=321)
b1,_ = data.synthetic_daily(n_years=33, tide_bps=12.0, seed=321)
c0, c1 = st.calendar_contrast(b0), st.calendar_contrast(b1)
ctrl = pd.DataFrame({
    'planted tide (bps/day)': [0, 12],
    'detected diff (bps/day)': [c0['diff_bps'], c1['diff_bps']],
    'HAC t': [c0['tstat_diff'], c1['tstat_diff']],
})
display(ctrl)
print('Clears the bar when a tide is real; insignificant when none is planted ->'
      ' the real-tape t=1.27 is a property of the market, not a blind pipeline.')


,planted tide (bps/day),detected diff (bps/day),HAC t
0,0,-0.712,-0.267
1,12,11.288,4.236


Clears the bar when a tide is real; insignificant when none is planted -> the real-tape t=1.27 is a property of the market, not a blind pipeline.
